# MRI H5 Metadata Exploration

This notebook explores and summarizes metadata from MRI HDF5 (H5) files. It includes loading k-space data, parsing XML metadata, statistical analysis, and visualization of both metadata and k-space data.

In [11]:
# get home folder path
import os
import random
import ismrmrd
import h5py
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import xml.dom.minidom as minidom
import xml.dom.minidom
from collections import Counter
import torch

home_folder = os.path.expanduser("~")
parent_folder = os.path.abspath(os.path.join(home_folder, "..", ".."))
# cd to data folder
os.chdir(parent_folder)

data_folder = Path(parent_folder) / "data/datasets/msk_mri_h5/test_varnet_mrpro"

h5_paths = sorted(
    p for p in data_folder.iterdir()
    if p.suffix.lower() in {'.h5', '.hdf5'} and p.is_file()
)
print(f'Found {len(h5_paths)} H5 files in: {data_folder}')

# choose random file from folder
random_file = random.choice(h5_paths)
#random_file = "meas_MID00017_FID126574_RT_3_PLANE_UOF.h5"  # specific file for testing
print(f"Random file chosen from the data folder: {random_file}")

folder_size = sum(f.stat().st_size for f in data_folder.glob('**/*') if f.is_file()) / (1024 ** 3)
print(f"Size of the data folder: {folder_size:.2f} GB")

Found 13 H5 files in: /data/datasets/msk_mri_h5/test_varnet_mrpro
Random file chosen from the data folder: /data/datasets/msk_mri_h5/test_varnet_mrpro/meas_MID00043_FID116514_AX_T2_ANGLE_DISC.h5
Size of the data folder: 26.12 GB


In [12]:
with h5py.File(os.path.join(data_folder, random_file), 'r') as f:
    for key in f.keys():
        print(f"{key}: {f[key].shape} - {f[key].dtype}")

ismrmrd_header: () - |V15680
kspace: (48, 12, 404, 640) - complex64
reconstruction_rss: (48, 320, 320) - float32


## Load K-space and Print Metadata Function

Defines a function to load k-space data from an HDF5 file using ISMRMRD, optionally printing the XML metadata structure. Loads k-space data for the selected file.

In [ ]:
# print_ismrmrd_header_nice.py
import os, re, json, codecs
from pathlib import Path

import h5py
import numpy as np
import xmltodict

# ----------------- utils -----------------
def _coerce_bytes(x):
    if isinstance(x, (bytes, bytearray)): return bytes(x)
    if isinstance(x, np.ndarray): return x.tobytes()
    return bytes(x)

def _bytes_human(n):
    for unit in ("B","KB","MB","GB","TB"):
        if n < 1024: return f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} PB"

# ----------------- ISMRMRD XML -----------------
def read_ismrmrd_xml(path):
    with h5py.File(path, "r") as f:
        raw = f["ismrmrd_header"][()]
    buf = _coerce_bytes(raw).replace(b"\x00", b"").lstrip(codecs.BOM_UTF8)
    s = buf.decode("utf-8", errors="ignore")
    if "<" in s and ">" in s:
        s = s[s.find("<"): s.rfind(">")+1]
    return s.strip()

def strip_xml_namespaces(xml_text: str) -> str:
    # remove xmlns decls and prefixes to make the dict cleaner
    xml_text = re.sub(r'\s+xmlns(:\w+)?="[^"]*"', '', xml_text)
    xml_text = re.sub(r'</\s*([\w\-]+):', r'</', xml_text)
    xml_text = re.sub(r'<\s*([\w\-]+):', r'<', xml_text)
    return xml_text

def xml_to_dict(xml_text: str):
    return xmltodict.parse(xml_text, dict_constructor=dict)

def print_nice(obj):
    try:
        import yaml
        print(yaml.safe_dump(obj, sort_keys=False, allow_unicode=True))
    except Exception:
        print(json.dumps(obj, indent=2, ensure_ascii=False))

# ----------------- kspace / rss loaders -----------------
def load_kspace(path) -> np.ndarray:
    """Returns full k-space as a NumPy array from '/kspace'."""
    with h5py.File(path, "r") as f:
        if "kspace" not in f:
            raise KeyError("Dataset 'kspace' not found.")
        return f["kspace"][()]  # loads to memory

def load_rss(path) -> np.ndarray:
    """Returns full RSS reconstruction as a NumPy array from '/reconstruction_rss'."""
    with h5py.File(path, "r") as f:
        if "reconstruction_rss" not in f:
            raise KeyError("Dataset 'reconstruction_rss' not found.")
        return f["reconstruction_rss"][()]  # loads to memory

def print_array_info(name: str, arr: np.ndarray):
    size_bytes = arr.nbytes
    print(f"{name}: shape={arr.shape}, dtype={arr.dtype}, size={_bytes_human(size_bytes)}")

path = Path(os.path.join(data_folder, random_file))  # provided by your notebook
xml_raw = read_ismrmrd_xml(path)
xml_clean = strip_xml_namespaces(xml_raw)
data = xml_to_dict(xml_clean)
print_nice(data)

try:
    ksp = load_kspace(path)
    print_array_info("kspace", ksp)
except Exception as e:
    print(f"kspace: not available ({e})")

try:
    rss = load_rss(path)
    print_array_info("reconstruction_rss", rss)
except Exception as e:
    print(f"reconstruction_rss: not available ({e})")




ismrmrdHeader:
  '@xsi:schemaLocation': http://www.ismrm.org/ISMRMRD ismrmrd.xsd
  subjectInformation:
    patientName: xxxxxxxxxxxxxxxxxxxxxx
    patientWeight_kg: '57.1529999'
    patientHeight_m: '1626'
    patientID: 18.0.222663743
    patientGender: F
  studyInformation:
    studyTime: 07:34:03
    studyID: '222663751'
  measurementInformation:
    measurementID: '169597_222663743_222663751_43'
    patientPosition: HFS
    protocolName: AX T2 ANGLE DISC
    measurementDependency:
    - dependencyType: SenMap
      measurementID: '169597_222663743_222663751_33'
    - dependencyType: Noise
      measurementID: '169597_222663743_222663751_33'
    frameOfReferenceUID: 1.3.12.2.1107.5.2.41.169597.1.20250731071918167.0.0.4997
  acquisitionSystemInformation:
    systemVendor: SIEMENS
    systemModel: Avanto_fit
    systemFieldStrength_T: '1.49399996'
    relativeReceiverNoiseBandwidth: '0.792999983'
    receiverChannels: '12'
    coilLabel:
    - coilNumber: '22'
      coilName: Spine_32